### Variational Autoencoder
> following https://mbernste.github.io/posts/vae/

<img src="./images/drawing.png" alt="drawing" width="50%"/>


In [1]:
import torch
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import numpy as np
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
print(device)

cuda


In [2]:
from torch.utils.data import TensorDataset, DataLoader

batch_size = 32

def read_idx3_images(path):
    with open(path, 'rb') as f:
        # skip the 16 byte header: magic number(4), count(4), rows(4), cols(4)
        data = np.fromfile(f, dtype=np.uint8, offset=16)
    # reshape to (#images, 784) 
    return data.reshape(-1, 784).astype(np.float32) / 255.0

train_images = read_idx3_images('train-images.idx3-ubyte')

x_train = torch.tensor(train_images, dtype=torch.float32, device=device)

train_ds = TensorDataset(x_train, x_train)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, )
print(x_train.shape)
print(f"1 epoch = {10000/batch_size} batches")


torch.Size([60000, 784])
1 epoch = 312.5 batches


$\text{ELBO}(\phi, \theta) = \sum_{i=1}^{n} E_{z_{i} \sim q} \left[\log p_{\theta}\left(x_{i} | z_{i}\right) - KL \left(q\left(z_{i} | x_{i}\right) \parallel p(z_{i})\right)\right]$

$KL \left(q\left(z_{i} | x_{i}\right) \parallel p(z_{i})\right) = -\frac{1}{2}\sum_{j=1}^{J} \left(1 + \log \sigma_{j}^{2} - \mu_{j}^{2} - e^{ \log \sigma_{j}^{2}}  \right)$

In [11]:
# hyperparams
@dataclass
class VAEConfig:
    input_dim: int = 784
    hidden_dim: int = 512
    hidden_layers: int = 2
    latent_channels: int = 32
    lr: float = 3e-4

class FFN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        
        self.layers = nn.Sequential(
            nn.Linear(input_dim, output_dim),
            nn.ReLU()
        )
        
    def forward(self, x):
        return self.layers(x)


class VAE(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        assert config.hidden_dim % (2**config.hidden_layers) == 0, "not divisible by layers"
        
        self.encode = nn.Sequential(
            FFN(config.input_dim, config.hidden_dim),
            *[FFN(config.hidden_dim // (2 ** i), config.hidden_dim // (2 ** (i+1))) for i in range(config.hidden_layers)]
        )

        z_dim = config.hidden_dim // (2 ** config.hidden_layers)

        self.mulayer = FFN(z_dim, config.latent_channels)
        self.logvar = FFN(z_dim, config.latent_channels)

        
        self.decode = nn.Sequential(
            FFN(config.latent_channels, z_dim),
            *[FFN(z_dim * (2 ** i), z_dim*(2 ** (i+1))) for i in range(config.hidden_layers)],
            nn.Linear(config.hidden_dim, config.input_dim), # proj back up
        )

    def forward(self, x, targets=None, beta=1.0):
        x = self.encode(x)
        mu = self.mulayer(x)
        logvar = self.logvar(x)
        
        reparam = mu + torch.randn_like(logvar) * torch.exp(0.5 * logvar)
        output = self.decode(reparam)
        
        if targets is None:
            loss = None
        else:
            bce_loss = F.binary_cross_entropy_with_logits(output, targets, reduction="sum") / batch_size
            # mse_loss = F.mse_loss(input=output, target=targets, reduction="sum") / batch_size
            kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / batch_size
            
            loss = bce_loss + (beta / self.config.input_dim) * kl_loss
        return output, loss, bce_loss, kl_loss


small note. either mse loss / batch size adn then kl is sum

In [12]:
def get_lr(iter):
    return 3e-4 - (6e-9 * iter)  

In [13]:
model = VAE(VAEConfig()).to(device)

model.load_state_dict(torch.load("vaemnist.pt", weights_only=True))

optimizer = torch.optim.AdamW(model.parameters(), lr=model.config.lr)
expected, outputs = [], []
divergence_warmup = 10000
step = 0

In [14]:
# for epoch in range(2):
while (step < 20000):
    for batch_idx, (images, labels) in enumerate(train_loader):
        optimizer.zero_grad(set_to_none=True)
        
        with torch.autocast(device_type=device, dtype=torch.bfloat16):
            output, loss, recon_loss, kl_loss = model(images, labels)

        loss.backward()
        optimizer.step()
        
        
        lr = get_lr(step)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr
        
        
        if (step % 1000 == 0):
            print(f"step: {step} | loss: {loss.item():.2f} {recon_loss.item():.2f} {kl_loss.item():.2f} | lr: {lr:.6f} | beta: {min(1, step / divergence_warmup)}")
        step += 1
    outputs.append(output.to(torch.float32))
    expected.append(labels.to(torch.float32))



step: 0 | loss: 546.75 546.74 6.89 | lr: 0.000300 | beta: 0.0
step: 1000 | loss: 102.53 100.90 1276.08 | lr: 0.000294 | beta: 0.1
step: 2000 | loss: 91.59 89.76 1434.83 | lr: 0.000288 | beta: 0.2
step: 3000 | loss: 84.51 82.61 1487.25 | lr: 0.000282 | beta: 0.3
step: 4000 | loss: 90.19 88.24 1530.30 | lr: 0.000276 | beta: 0.4
step: 5000 | loss: 89.72 87.39 1825.54 | lr: 0.000270 | beta: 0.5
step: 6000 | loss: 86.41 84.02 1880.23 | lr: 0.000264 | beta: 0.6
step: 7000 | loss: 79.60 77.38 1737.75 | lr: 0.000258 | beta: 0.7
step: 8000 | loss: 78.19 75.88 1811.62 | lr: 0.000252 | beta: 0.8
step: 9000 | loss: 69.72 67.40 1817.32 | lr: 0.000246 | beta: 0.9
step: 10000 | loss: 72.91 70.61 1801.74 | lr: 0.000240 | beta: 1
step: 11000 | loss: 74.70 72.38 1824.65 | lr: 0.000234 | beta: 1
step: 12000 | loss: 78.14 76.05 1636.51 | lr: 0.000228 | beta: 1
step: 13000 | loss: 75.30 72.76 1992.17 | lr: 0.000222 | beta: 1
step: 14000 | loss: 75.49 73.25 1756.34 | lr: 0.000216 | beta: 1
step: 15000 | los

In [20]:
torch.save(model.state_dict(), "vaemnist.pt")

In [19]:
import matplotlib.pyplot as plt

# img = Image.fromarray((arr * 255).clip(0, 255).astype(np.uint8), mode="L")
# img.save("reconstructed.png")

from ipywidgets import interact, Layout, IntSlider
def show_image(i):
    batch = i//batch_size
    print(f"epoch {batch/625}, batch {batch}, total examples: {i}")
    predicted = outputs[batch][i % batch_size].cpu().detach().reshape(28, 28).numpy() # 16 is batch
    e = expected[batch][i % batch_size].cpu().detach().reshape(28, 28).numpy()
    
    plt.figure(figsize=(6, 3))
    plt.subplot(1, 2, 1)
    
    plt.imshow(predicted.squeeze(), cmap="gray", vmin=0, vmax=1)
    plt.subplot(1, 2, 2)
    plt.imshow(e, cmap="gray")
    plt.show()
    
interact(show_image, i=IntSlider(min=0, max=batch_size*(len(outputs)) - 1, layout=Layout(width='75%')))

interactive(children=(IntSlider(value=0, description='i', layout=Layout(width='75%'), max=351), Output()), _do…

<function __main__.show_image(i)>